In [ ]:
%pip install opencv-python transformers Pillow sentencepiece protobuf 

In [ ]:
import os
import cv2
import torch
from dotenv import load_dotenv
from PIL import Image
from transformers import AutoProcessor, AutoModel

# 1. Load SigLIPv2 mmodel
print("Loading SigLIPv2 model...")
load_dotenv()
my_hf_token = os.getenv("HF_TOKEN")
model_id = "google/siglip-so400m-patch14-384"
processor = AutoProcessor.from_pretrained(model_id, token = my_hf_token)
model = AutoModel.from_pretrained(model_id, token = my_hf_token)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# 2. Define events to search
labels = [
    "a photo of a rugby scrum", 
    "a photo of a rugby tackle", 
    "a photo of a rugby ruck",      
    "a photo of a rugby maul",       
    "a photo of a rugby lineout",
    "a photo of a rugby referee",
    "a photo of a rugby try",
    "a photo of a rugby player kicking the ball",
    "a photo of a rugby player passing the ball by hand"
]

# 3. Open video filefile
video_path = "FR-EN.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print(f"Can not open video at: {video_path}")
    exit()

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / fps
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Video length: {duration:.2f} secs | FPS: {fps:.2f} | Resolution: {width}x{height}")

# 4. Setup Video Writer
output_path = "output_annotated.mp4"
# MP4V codec for MP4 format
fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# Scanning configuration: 1.0 FPS to save time 
frame_interval = int(fps * 1) 
threshold = 0.15 

print("\n--- Analyzing and Generating Video ---")

frame_count = 0
current_detected_events = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    # Process frames by circle
    if frame_count % frame_interval == 0:
        timestamp = frame_count / fps
        
        # OpenCV BGR to RGB for PIL
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(rgb_frame)
        
        # Put through SigLIP
        inputs = processor(text=labels, images=pil_image, padding="max_length", return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()} # Data to GPU
        
        with torch.no_grad():
            outputs = model(**inputs)
            
        # Sigmoid probability
        probs = torch.sigmoid(outputs.logits_per_image)[0]
        
        # Update current events
        current_detected_events = []
        for i, prob in enumerate(probs):
            if prob.item() > threshold:
                clean_label = labels[i].replace("a photo of ", "") 
                current_detected_events.append(f"{clean_label}: {prob.item():.1%}")
                
        if current_detected_events:
            time_str = f"{int(timestamp//60):02d}:{int(timestamp%60):02d}"
            print(f"[{time_str}] Detected: " + " | ".join(current_detected_events))

    # Draw labels on frame
    y_offset = 40
    for event_text in current_detected_events:
        (text_width, text_height), baseline = cv2.getTextSize(event_text, cv2.FONT_HERSHEY_SIMPLEX, 1, 2)
        cv2.rectangle(frame, (20, y_offset - text_height - 10), (20 + text_width + 20, y_offset + baseline), (0, 0, 0), -1)
        
        cv2.putText(frame, event_text, (30, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        y_offset += text_height + 20

    out.write(frame)
    frame_count += 1

cap.release()
print("--- Done ---")

Đang tải mô hình SigLIP...


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

Video dài: 782.60 giây | FPS: 25.00

--- BẮT ĐẦU PHÂN TÍCH ---
[00:00] Phát hiện: a rugby ruck (15.5%) | a rugby referee signaling a penalty (65.9%) | a rugby player passing the ball (44.8%)
[00:01] Phát hiện: a rugby player kicking the ball (15.5%) | a rugby player passing the ball (53.7%) | a player scoring a rugby try (42.2%)
[00:02] Phát hiện: a rugby player passing the ball (19.0%) | a player scoring a rugby try (31.3%)
[00:03] Phát hiện: a rugby tackle (22.7%) | a rugby player passing the ball (42.2%) | a player scoring a rugby try (56.5%)
[00:06] Phát hiện: a rugby tackle (16.6%) | a rugby ruck (51.5%) | a rugby lineout (44.7%)
[00:07] Phát hiện: a rugby ruck (76.3%) | a rugby lineout (92.1%) | a rugby referee signaling a penalty (22.5%) | a rugby player passing the ball (16.9%)
[00:08] Phát hiện: a rugby ruck (78.5%) | a rugby maul (15.1%) | a rugby lineout (91.5%) | a rugby referee signaling a penalty (62.1%) | a rugby player passing the ball (21.2%)
[00:09] Phát hiện: a rugby